# Gene level table

One row per disease-associated gene. `uniqueDiseases` is the gene-level pleiotropy score
(gPS): the number of distinct diseases associated with any variant sharing that gene as
likely causal. `uniqueTherapeuticAreas` is the same count collapsed to therapeutic areas.
Gene properties (constraint, pathway count, tissue specificity) come from the release
target datasets. Methods "Gene-level pleiotropy modelling".

Writes `gene_table`.

In [1]:
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 00:28:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
genes = session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
studies = session.spark.read.parquet(paper.derived("study_therapeutic_areas")).select(
    "studyId", "mappedTherapeuticAreas", *paper.TA_COLUMNS.values()
)
rows = genes.join(studies, "studyId", "inner")
print("gene x credible-set rows:", rows.count())

gene x credible-set rows: 70400


## Aggregate to genes

In [3]:
per_gene = rows.groupBy("geneId").agg(
    f.size(f.array_distinct(f.flatten(f.collect_list("diseaseIds")))).alias("uniqueDiseases"),
    f.size(f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas")))).alias("uniqueTherapeuticAreas"),
    f.max("eQTL_coloc").alias("maxEQTLColoc"),
    f.max("pQTL_coloc").alias("maxPQTLColoc"),
    f.max("VEP").alias("maxVEP"),
    f.max("distanceTSS").alias("maxDistanceTSS"),
    f.min("effectiveSampleSize").alias("minEffectiveSampleSize"),
    f.max("effectiveSampleSize").alias("maxEffectiveSampleSize"),
    f.min("publicationDate").alias("earliestPublicationDate"),
    f.countDistinct("variantId").alias("uniqueVariants"),
    f.max("absBeta").alias("maxAbsBeta"),
    f.min("maf").alias("minMaf"),
    *[f.sum(column).alias(column) for column in paper.TA_COLUMNS.values()],
)
per_gene = per_gene.withColumn("totalStudies", sum(f.col(c) for c in paper.TA_COLUMNS.values()))
print("genes:", per_gene.count())

genes: 8285


## Gene properties

In [4]:
target = session.spark.read.parquet(paper.release("target"))

properties = (
    target.withColumns(
        {
            "lofConstraint": f.filter("constraint", lambda x: x.constraintType == "lof")[0]["oeUpper"],
            "misConstraint": f.filter("constraint", lambda x: x.constraintType == "mis")[0]["score"],
            "synConstraint": f.filter("constraint", lambda x: x.constraintType == "syn")[0]["score"],
        }
    )
    .select(
        f.col("id").alias("geneId"),
        "approvedSymbol",
        "biotype",
        (1 - f.col("lofConstraint")).alias("lofConstraint"),
        "misConstraint",
        "synConstraint",
        f.size("pathways").alias("pathwayCount"),
        (f.col("genomicLocation.end") - f.col("genomicLocation.start")).alias("geneLength"),
    )
    .withColumn("pathwayCount", f.when(f.col("pathwayCount") == -1, 0).otherwise(f.col("pathwayCount")))
)

gene_table = per_gene.join(properties, "geneId", "inner")
print("genes with target properties:", gene_table.count())

genes with target properties: 8285


## Tissue specificity

`target_prioritisation` is part of the release but is not on disk (GAPS.md item 1). Where it
is absent the two columns are written as null and the gPS tissue-specificity covariate
cannot be fitted; everything else in this table is unaffected.

In [5]:
from pathlib import Path

prioritisation = Path(paper.release("target_prioritisation"))
if prioritisation.exists():
    tissue = session.spark.read.parquet(str(prioritisation)).select(
        f.col("targetId").alias("geneId"), "tissueSpecificity", "tissueDistribution", "hasSafetyEvent"
    )
    gene_table = gene_table.join(tissue, "geneId", "left")
else:
    print("target_prioritisation missing: tissueSpecificity, tissueDistribution, hasSafetyEvent are null")
    gene_table = (
        gene_table.withColumn("tissueSpecificity", f.lit(None).cast("double"))
        .withColumn("tissueDistribution", f.lit(None).cast("double"))
        .withColumn("hasSafetyEvent", f.lit(None).cast("double"))
    )

gene_table.write.mode("overwrite").parquet(paper.derived("gene_table"))
gene_table = session.spark.read.parquet(paper.derived("gene_table"))
print("gene_table rows:", gene_table.count())

26/08/19 00:28:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


gene_table rows: 8285


## Cross-check against the pre-refactor table

In [6]:
new = gene_table.select("geneId", "uniqueDiseases", "uniqueTherapeuticAreas").toPandas()
ref = (
    session.spark.read.parquet(paper.baseline("genes_therapeutic_areas"))
    .select("geneId", "uniqueDiseases", "uniqueTherapeuticAreas")
    .toPandas()
)
merged = new.merge(ref, on="geneId", how="outer", suffixes=("_new", "_baseline"), indicator=True)
print(merged["_merge"].value_counts().to_string())
both = merged[merged["_merge"] == "both"]
print("genes with a different gPS:", (both["uniqueDiseases_new"] != both["uniqueDiseases_baseline"]).sum())
print(
    "genes with a different TA count:",
    (both["uniqueTherapeuticAreas_new"] != both["uniqueTherapeuticAreas_baseline"]).sum(),
)

_merge
both          8285
left_only        0
right_only       0
genes with a different gPS: 0
genes with a different TA count: 0
